In [24]:
import json
from pathlib import Path

from os import listdir

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eigh_tridiagonal

from physics_definitions import load_physics_definitions


In [25]:
ROOT = Path().cwd().parent

In [26]:
SOURCE_MANIFEST = ROOT / "plot_potentials" / "graphs_ID" / "manifest.json"
OUTPUT_DIR = ROOT / "plot_potentials" / "Graph_ID_V+En"
K_STATES = 11
NODES = 2049
KINETIC_COEFFICIENT = 0.5

In [27]:
def solve_uniform_dirichlet(x: np.ndarray, potential: np.ndarray):
    """Second-order Dirichlet discretization of -kappa*d2/dx2 + V."""
    dx = float(x[1] - x[0])
    diagonal = np.full(len(x) - 2, 2.0 * KINETIC_COEFFICIENT / dx**2) + potential[1:-1]
    off_diagonal = np.full(len(x) - 3, -KINETIC_COEFFICIENT / dx**2)
    energy, interior = eigh_tridiagonal(
        diagonal, off_diagonal, select="i", select_range=(0, K_STATES - 1), check_finite=True
    )
    psi = np.zeros((K_STATES, len(x)))
    psi[:, 1:-1] = interior.T
    for state in range(K_STATES):
        psi[state] /= np.sqrt(np.trapezoid(psi[state] ** 2, x))
        pivot = int(np.argmax(np.abs(psi[state])))
        if psi[state, pivot] < 0:
            psi[state] *= -1
    return energy, psi

In [28]:
PHYSICS_STAGE = ROOT / "stages" / "s01_physics.py"
SELECTED_NAMES = {
    "ANALYTICAL_FAMILIES",
    "FUNCTIONAL_FAMILIES",
    "TRAIN_FAMILIES",
    "REPARAMETERIZED_SEEN_FAMILIES",
    "UNSEEN_ANALYTICAL_FAMILIES",
    "HELD_OUT_FAMILIES",
    "PotentialSpec",
    "_coefficients",
    "make_spec",
    "potential_value",
    "geometry_grid",
    "sample_potential_safely",
}

In [29]:
definitions = load_physics_definitions(PHYSICS_STAGE, SELECTED_NAMES)
PotentialSpec = definitions["PotentialSpec"]
sample_potential_safely = definitions["sample_potential_safely"]
records = json.loads(SOURCE_MANIFEST.read_text(encoding="utf-8"))

In [30]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_manifest_path = OUTPUT_DIR / "manifest.json"
output_manifest = (
    json.loads(output_manifest_path.read_text(encoding="utf-8"))
    if output_manifest_path.exists() else []
)
completed = {item["family"] for item in output_manifest}

In [35]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 16,
    "axes.titlesize": 25,
    "axes.labelsize": 25,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

In [38]:
for index, record in enumerate(records, 1):
    """
    if record["family"] in completed:
        print(f"Skipping existing {record['family']}")
        continue
    """
    left, right = map(float, record["domain"])
    family = record["family"]
    singular_left = (
        family in {"poschl_t", "radial_coulomb", "kratzer"}
        or (family == "radial_oscillator" and int(record["parameters"].get("ell", 0)) > 0)
    )
    singular_right = family == "poschl_t"
    spec = PotentialSpec(
        family, record["parameters"], record["geometry"],
        "friedrichs_dirichlet" if singular_left or singular_right else "dirichlet",
        left, right, None, singular_left, singular_right, "id_visualization", f"id-plot-{index}",
    )
    x = np.linspace(left, right, NODES)
    potential, valid, _ = sample_potential_safely(spec, x)
    energy, psi = solve_uniform_dirichlet(x, potential)

    gaps = np.diff(energy)
    amplitude = 0.34 * float(np.median(gaps)) / max(float(np.max(np.abs(psi))), 1e-12)
    colors = plt.colormaps["turbo"](np.linspace(0.04, 0.96, K_STATES))
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(x[valid], potential[valid], color="black", linewidth=2.2, label=r"$V(x)$", zorder=3)
    for state, (level, wave, color) in enumerate(zip(energy, psi, colors)):
        ax.axhline(level, color=color, linewidth=0.65, linestyle="--", alpha=0.45)
        ax.plot(x, level + amplitude * wave, color=color, linewidth=1.35,
                label=rf"$n={state}$, $E={level:.4f}$")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"Energy and $E_n + A\psi_n(x)$")
    ax.set_title(f"{record['family'].replace('_', ' ').title()}: potential and first {K_STATES} eigenstates")
    ax.grid(alpha=0.16)
    #ax.legend(ncol=2, fontsize=8, loc="upper left")
    top = float(energy[-1] + 0.75 * np.median(gaps))
    visible_potential_floor = float(np.quantile(potential[valid], 0.01))
    bottom = min(visible_potential_floor, float(energy[0] - 0.75 * np.median(gaps)))
    ax.set_ylim(bottom, top)
    fig.tight_layout()
    filename = f"{index:02d}_{record['family']}_V_plus_eigenstates.png"
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.close(fig)
    output_manifest.append({
        "file": filename, "family": record["family"], "parameters": record["parameters"],
        "domain": record["domain"], "states": list(range(K_STATES)),
        "energies": energy.tolist(), "nodes": NODES,
        "operator": "-0.5*d2/dx2 + V(x)", "boundary_condition": "Dirichlet",
        "wavefunction_display_scale": amplitude,
    })
    output_manifest_path.write_text(json.dumps(output_manifest, indent=2), encoding="utf-8")


In [37]:
output_manifest.sort(key=lambda item: item["file"])
output_manifest_path.write_text(json.dumps(output_manifest, indent=2), encoding="utf-8")
print(f"Manifest contains {len(output_manifest)} potential/eigenstate plots in {OUTPUT_DIR}")


Manifest contains 189 potential/eigenstate plots in /home/clouduser/psi_Jepa_V08/plot_potentials/Graph_ID_V+En


In [34]:
output_manifest

[{'file': '01_infinite_box_V_plus_eigenstates.png',
  'family': 'infinite_box',
  'parameters': {'L': 11.499308526800588},
  'domain': [0.0, 11.499308526800588],
  'states': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
  'energies': [0.03731867468432212,
   0.1492746109223756,
   0.33586754526766116,
   0.5970970386517767,
   0.9329624763719171,
   1.343463068108481,
   1.8285978479162655,
   2.3883656742244694,
   3.0227652298454943,
   3.731795021974947,
   4.515453382191636],
  'nodes': 2049,
  'operator': '-0.5*d2/dx2 + V(x)',
  'boundary_condition': 'Dirichlet',
  'wavefunction_display_scale': 0.36509091299208113},
 {'file': '01_infinite_box_V_plus_eigenstates.png',
  'family': 'infinite_box',
  'parameters': {'L': 11.499308526800588},
  'domain': [0.0, 11.499308526800588],
  'states': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
  'energies': [0.03731867468432212,
   0.1492746109223756,
   0.33586754526766116,
   0.5970970386517767,
   0.9329624763719171,
   1.343463068108481,
   1.828597847916265